# 10. Graph-Based Topic Modeling via FAISS and Leiden Community Detection

This notebook implements a topology-based topic modeling pipeline using embeddings fine-tuned with PRISM (CoSENT loss). Since the embeddings are semantically aligned via cosine distance, we construct an exact k-NN graph using FAISS, followed by the Leiden algorithm (Constant Potts Model) to isolate highly cohesive semantic communities.

## 10.1 Environment Setup

In [1]:
import importlib.util
import subprocess
import sys

def check_and_install(package_name, pip_name=None):
    """
    Checks if a package is installed and attempts to install it via pip in Colab environments.
    Prints a warning for local environments.
    
    Args:
        package_name (str): The name of the module to check.
        pip_name (str, optional): The name of the package on PyPI. Defaults to package_name.
    """
    if pip_name is None:
        pip_name = package_name
    if importlib.util.find_spec(package_name) is None:
        try:
            from google.colab import drive
            print(f"Installing {pip_name} in Colab...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        except ImportError:
            print(f"Warning: '{package_name}' is missing. Make sure to install '{pip_name}' in your local .venv")

check_and_install("faiss", "faiss-cpu")
check_and_install("igraph")
check_and_install("leidenalg")

In [2]:
import os
import numpy as np
import pandas as pd
import faiss
import igraph as ig
import leidenalg as la
from tqdm.auto import tqdm
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    BASE_PATH = Path("/content/drive/Shareddrives/Minería/proyecto_horus/")
else:
    BASE_PATH = Path("../")

# Constants
EMBEDDINGS_PATH = BASE_PATH / "data/processed/prism_embeddings.npy"
METADATA_PATH = BASE_PATH / "data/processed/products_modeling.parquet"
OUTPUT_PATH = BASE_PATH / "data/processed/graph_topics.parquet"

# Grid Search will dynamically explore these parameters
MAX_K_NEIGHBORS = 50

## 10.2 Data Loading & Preparation

We load the 1024D float32 PRISM embeddings and the corresponding dataset metadata. The embeddings are L2 normalized to ensure FAISS inner product search mathematically equates to cosine similarity.

In [3]:
# Load Data
print(f"Loading metadata from {METADATA_PATH}...")
df_meta = pd.read_parquet(METADATA_PATH)

print(f"Loading embeddings from {EMBEDDINGS_PATH}...")
embeddings = np.load(EMBEDDINGS_PATH)

print(f"Embeddings shape: {embeddings.shape}, dtype: {embeddings.dtype}")
print(f"Metadata shape: {df_meta.shape}")

# L2 Normalize embeddings for Cosine Similarity (Inner Product)
print("Normalizing embeddings for Cosine Similarity...")
faiss.normalize_L2(embeddings)

Loading metadata from ../data/processed/products_modeling.parquet...
Loading embeddings from ../data/processed/prism_embeddings.npy...
Embeddings shape: (99064, 1024), dtype: float32
Metadata shape: (99064, 16)
Normalizing embeddings for Cosine Similarity...


## 10.3 FAISS Exact K-NN Graph Construction

We build an exact k-NN graph. Since the dataset is ~100k vectors, an exact index (`IndexFlatIP`) fits perfectly in RAM and is extremely fast on CPU. We query the top `K=15` neighbors.

In [4]:
d = embeddings.shape[1]  # 1024
index = faiss.IndexFlatIP(d)

print("Adding vectors to FAISS index...")
index.add(embeddings)
print(f"Total vectors in index: {index.ntotal}")

print(f"Searching top {MAX_K_NEIGHBORS} nearest neighbors (for subsequent Grid Search)...")
distances, indices = index.search(embeddings, MAX_K_NEIGHBORS)

print(f"Search complete. Distances shape: {distances.shape}")

Adding vectors to FAISS index...
Total vectors in index: 99064
Searching top 50 nearest neighbors (for subsequent Grid Search)...
Search complete. Distances shape: (99064, 50)


## 10.4 Graph Construction Helper

We define a function to build the igraph object dynamically given `K_NEIGHBORS` and `MIN_SIMILARITY`. 
By slicing the precomputed FAISS indices, we avoid re-running the expensive nearest neighbor search.

In [5]:
def build_graph(k, min_sim):
    k_indices = indices[:, :k]
    k_distances = distances[:, :k]
    num_nodes = embeddings.shape[0]
    
    # We use a dict for edges to automatically handle duplicates from bidirectional KNN
    edge_dict = {}
    
    for i in tqdm(range(num_nodes), desc=f"Building Edges (K={k}, Sim={min_sim})", leave=False):
        for j_idx, neighbor in enumerate(k_indices[i]):
            similarity = k_distances[i, j_idx]
            
            if i == neighbor:
                continue
                
            if similarity >= min_sim:
                source, target = sorted([i, neighbor])
                # Using a dict to keep the highest similarity in case of slight numerical differences
                edge = (source, target)
                if edge not in edge_dict or similarity > edge_dict[edge]:
                    edge_dict[edge] = float(similarity)
                    
    unique_edges = list(edge_dict.keys())
    unique_weights = list(edge_dict.values())
    
    G = ig.Graph(n=num_nodes, edges=unique_edges, directed=False)
    G.es['weight'] = unique_weights
    
    return G

## 10.5 Leiden Community Detection - 3D Grid Search

We perform a 3D grid search over `K_NEIGHBORS`, `MIN_SIMILARITY`, and `RESOLUTION_PARAMETER` 
to find a "sweet spot" where the number of communities is reasonable (e.g. 50-300).

In [6]:
import time

print("Running 3D Grid Search for Graph & Leiden Parameters...")

# Define search space
k_values = [15, 30, 50]
sim_values = [0.70, 0.75, 0.80]
res_values = [0.001, 0.005, 0.01, 0.05, 0.1, 0.2]

results = []

for k in k_values:
    for sim in sim_values:
        print(f"\n--- Building Graph for K={k}, Sim={sim} ---")
        start_graph = time.time()
        G = build_graph(k, sim)
        graph_time = time.time() - start_graph
        print(f"Graph built: {G.vcount()} nodes, {G.ecount()} edges (Took {graph_time:.2f}s)")
        
        for res in res_values:
            print(f"  Testing resolution_parameter = {res}...")
            start_leiden = time.time()
            partition = la.find_partition(
                G, 
                la.CPMVertexPartition, 
                weights=G.es['weight'],
                resolution_parameter=res
            )
            
            community_assignments = np.array(partition.membership)
            community_sizes = np.bincount(community_assignments)
            
            valid_comms = (community_sizes > 1).sum()
            outliers = (community_sizes == 1).sum()
            
            results.append({
                "K": k,
                "Sim": sim,
                "Resolution": res,
                "Modularity": partition.modularity,
                "Valid_Communities": valid_comms,
                "Outliers": outliers,
                "Total_Time (s)": round(graph_time + (time.time() - start_leiden), 2)
            })

results_df = pd.DataFrame(results)
display(results_df.sort_values(by="Modularity"))

Running 3D Grid Search for Graph & Leiden Parameters...

--- Building Graph for K=15, Sim=0.7 ---


Building Edges (K=15, Sim=0.7):   0%|          | 0/99064 [00:00<?, ?it/s]

Graph built: 99064 nodes, 1014382 edges (Took 3.30s)
  Testing resolution_parameter = 0.001...
  Testing resolution_parameter = 0.005...
  Testing resolution_parameter = 0.01...
  Testing resolution_parameter = 0.05...
  Testing resolution_parameter = 0.1...
  Testing resolution_parameter = 0.2...

--- Building Graph for K=15, Sim=0.75 ---


Building Edges (K=15, Sim=0.75):   0%|          | 0/99064 [00:00<?, ?it/s]

Graph built: 99064 nodes, 934476 edges (Took 3.40s)
  Testing resolution_parameter = 0.001...
  Testing resolution_parameter = 0.005...
  Testing resolution_parameter = 0.01...
  Testing resolution_parameter = 0.05...
  Testing resolution_parameter = 0.1...
  Testing resolution_parameter = 0.2...

--- Building Graph for K=15, Sim=0.8 ---


...

Building Edges (K=50, Sim=0.75):   0%|          | 0/99064 [00:00<?, ?it/s]

Graph built: 99064 nodes, 2637662 edges (Took 10.15s)
  Testing resolution_parameter = 0.001...
  Testing resolution_parameter = 0.005...
  Testing resolution_parameter = 0.01...
  Testing resolution_parameter = 0.05...
  Testing resolution_parameter = 0.1...
  Testing resolution_parameter = 0.2...

--- Building Graph for K=50, Sim=0.8 ---


Building Edges (K=50, Sim=0.8):   0%|          | 0/99064 [00:00<?, ?it/s]

Graph built: 99064 nodes, 1501088 edges (Took 7.37s)
  Testing resolution_parameter = 0.001...
  Testing resolution_parameter = 0.005...
  Testing resolution_parameter = 0.01...
  Testing resolution_parameter = 0.05...
  Testing resolution_parameter = 0.1...
  Testing resolution_parameter = 0.2...


,K,Sim,Resolution,Modularity,Valid_Communities,Outliers,Total_Time (s)
18,30,0.70,0.001,0.844869,134,237,35.40
36,50,0.70,0.001,0.843824,149,411,56.58
0,15,0.70,0.001,0.847141,197,132,20.07
37,50,0.70,0.005,0.785733,353,542,56.72
19,30,0.70,0.005,0.779257,366,381,35.90
6,15,0.75,0.001,0.864207,442,1053,20.99
24,30,0.75,0.001,0.864765,487,1523,33.54
38,50,0.70,0.010,0.741052,516,646,56.95
1,15,0.70,0.005,0.778075,595,163,19.92
20,30,0.70,0.010,0.734477,607,442,35.79


## 10.6 Final Community Assignment

Based on the grid search, we select the parameters that yield a useful number of valid communities 
(e.g. 50-300 topics) and assign them to the dataset.

In [25]:
BEST_K, BEST_SIM, BEST_RESOLUTION = results_df.loc[results_df['Modularity'].idxmax(), ['K', 'Sim', 'Resolution']]
# BEST_K, BEST_SIM, BEST_RESOLUTION = results_df.loc[results_df['Valid_Communities'].idxmin(), ['K', 'Sim', 'Resolution']]
BEST_K = int(BEST_K)

# BEST_K = 30             # Update this based on the grid search results
# BEST_SIM = 0.70         # Update this based on the grid search results
# BEST_RESOLUTION = 0.005 # Update this based on the grid search results
MODEL_SUFFIX = "max_modularity" # Change this to 'max_modularity', 'min_communities', etc.

print(f"Building Final Graph with K={BEST_K}, Sim={BEST_SIM}...")
G_final = build_graph(BEST_K, BEST_SIM)

print(f"Running Final Leiden CPM with resolution = {BEST_RESOLUTION}...")
partition = la.find_partition(
    G_final, 
    la.CPMVertexPartition, 
    weights=G_final.es['weight'],
    resolution_parameter=BEST_RESOLUTION
)

# Extract cluster IDs
community_assignments = np.array(partition.membership)

# Identify isolated nodes (degree 0) or communities of size 1 and label them as -1 (Outlier)
community_sizes = np.bincount(community_assignments)
outlier_communities = np.where(community_sizes == 1)[0]
community_assignments[np.isin(community_assignments, outlier_communities)] = -1

df_meta['community_id'] = community_assignments

# Renumber clusters so they are contiguous starting from 0, with -1 still representing outliers
unique_valid_clusters = sorted(set(community_assignments[community_assignments != -1]))
cluster_mapping = {old_id: new_id for new_id, old_id in enumerate(unique_valid_clusters)}
cluster_mapping[-1] = -1

df_meta['community_id'] = df_meta['community_id'].map(cluster_mapping)

total_valid_clusters = len(unique_valid_clusters)
print(f"Modularity: {partition.modularity:.4f}")
print(f"Total valid communities (size > 1): {total_valid_clusters}")
print(f"Total outliers (unconnected nodes): {(df_meta['community_id'] == -1).sum()}")

Building Final Graph with K=50, Sim=0.8...


Building Edges (K=50, Sim=0.8):   0%|          | 0/99064 [00:00<?, ?it/s]

Running Final Leiden CPM with resolution = 0.001...
Modularity: 0.9091
Total valid communities (size > 1): 2074
Total outliers (unconnected nodes): 6380


## 10.7 Aggregation & Cluster Profiling

We evaluate the top 10 largest communities, sampling titles to validate semantic cohesion.

In [26]:
print("Top 10 Largest Communities:")
top_communities = df_meta[df_meta['community_id'] != -1]['community_id'].value_counts().head(10)

for cluster_id, size in top_communities.items():
    print(f"\n========================================")
    print(f"Community ID: {cluster_id} | Size: {size}")
    print(f"========================================")
    samples = df_meta[df_meta['community_id'] == cluster_id]['original_title'].sample(n=min(5, size), random_state=42)
    for sample in samples:
        print(f"- {sample}")

Top 10 Largest Communities:

Community ID: 0 | Size: 3255
- Modelo De Distribución De Biomasa Seca En Plantulas De Maracuyá Encondicones De Invernadero
- Alimentos con sello campesino y las experiencias locales en las políticas públicas
- Metodología para evaluar la sostenibilidad de sistemas agrícolas de clima frío en cundinamarca y boyacá
- Fertirriego: memorias
- Efecto de cinco sustratos sobre índices de crecimiento de plantas de papaya (Carica papaya L.) bajo invernadero .........................................................................

Community ID: 1 | Size: 3057
- Multipartite entanglement in conditional states
- Cálculo de la expansión térmica de monocristales de kdp a partir de drx
- Hamiltoniano de fases magnéticas.
- Influence of Si on the Structural, Electrical, and Optical Properties of (Al, Ti, Si)N Films Deposited Via Reactive DC Sputtering
- Los óxidos cerámicos como materiales termoeléctricos

Community ID: 2 | Size: 2554
- Measurements of branching fractions 

## 10.8 Export

Save the resulting dataset with community assignments.

In [27]:
# ============================================================
# Export artifacts for Notebook 11 comparison
# Generates:
#   outputs/10_graph_top_terms.csv
#   outputs/10_graph_metrics.csv
# ============================================================

from sklearn.feature_extraction.text import CountVectorizer

OUTPUTS_DIR = BASE_PATH / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. TOP TERMS PER COMMUNITY
# ------------------------------------------------------------

print("Generating top terms per graph community...")

valid_df = df_meta[df_meta["community_id"] != -1].copy()

top_terms_rows = []

for community_id in sorted(valid_df["community_id"].unique()):

    docs = valid_df.loc[
        valid_df["community_id"] == community_id,
        "embeddings_text"
    ].dropna()

    if len(docs) < 2:
        continue

    try:
        vectorizer = CountVectorizer(
            stop_words="english",
            max_features=5000
        )

        X = vectorizer.fit_transform(docs)

        term_scores = np.asarray(X.sum(axis=0)).flatten()
        vocab = np.array(vectorizer.get_feature_names_out())

        top_idx = np.argsort(term_scores)[::-1][:10]

        top_terms_rows.append({
            "cluster_id": int(community_id),
            "top_terms": ", ".join(vocab[top_idx])
        })

    except Exception:
        continue

top_terms_df = pd.DataFrame(top_terms_rows)

TOP_TERMS_FILE = OUTPUTS_DIR / f"10_graph_top_terms_{MODEL_SUFFIX}.csv"
top_terms_df.to_csv(TOP_TERMS_FILE, index=False)

print(f"Saved: {TOP_TERMS_FILE}")
print(f"Topics exported: {len(top_terms_df)}")


# ------------------------------------------------------------
# 2. METRICS FOR NOTEBOOK 11
# ------------------------------------------------------------

print("Generating graph metrics...")

outlier_ratio = (
    (df_meta["community_id"] == -1).sum()
    / len(df_meta)
)

metrics_df = pd.DataFrame([{
    "model": f"graph_leiden_{MODEL_SUFFIX}",
    "modularity": float(partition.modularity),
    "n_topics": int(total_valid_clusters),
    "outlier_ratio": float(outlier_ratio)
}])

METRICS_FILE = OUTPUTS_DIR / f"10_graph_metrics_{MODEL_SUFFIX}.csv"

metrics_df.to_csv(METRICS_FILE, index=False)

print(f"Saved: {METRICS_FILE}")

display(metrics_df)

print("\nDone. Notebook 11 will now automatically detect:")
print(f" - 10_graph_top_terms_{MODEL_SUFFIX}.csv")
print(f" - 10_graph_metrics_{MODEL_SUFFIX}.csv")

OUTPUT_PATH_SUFFIXED = BASE_PATH / f"data/processed/graph_topics_{MODEL_SUFFIX}.parquet"
print(f"Exporting dataset with graph topics to {OUTPUT_PATH_SUFFIXED}...")
df_meta.to_parquet(OUTPUT_PATH_SUFFIXED)
print("Done!")

Generating top terms per graph community...
Saved: ../outputs/10_graph_top_terms_max_modularity.csv
Topics exported: 2074
Generating graph metrics...
Saved: ../outputs/10_graph_metrics_max_modularity.csv


,model,modularity,n_topics,outlier_ratio
0,graph_leiden_max_modularity,0.909096,2074,0.064403



Done. Notebook 11 will now automatically detect:
 - 10_graph_top_terms_max_modularity.csv
 - 10_graph_metrics_max_modularity.csv
Exporting dataset with graph topics to ../data/processed/graph_topics_max_modularity.parquet...
Done!
